# Loading the Data

In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

In [6]:
wec=pd.read_csv("C:\\Project DATA\\WEC Analysis 2021\\Data\\wec_hypercar_2021.csv")
LeMans=wec[wec['circuit']=='LE MANS']

# Analysis

## Driver Stint Consistency

In [ ]:
LeMans_filtered = LeMans[LeMans['pit_time'].isna() | (LeMans['pit_time'] == 0)]

fig = px.line(LeMans_filtered,
    x="lap_number",
    y="lap_time_s",
    title="Driver Stint Consistency",
    color="team_no",          
    line_group="driver_stint", 
    markers=True,
    labels={                  
        "lap_number": "Lap Number",
        "lap_time_s": "Lap Time (s)",
        "driver_stint": "Driver Stint",
        "team_no": "Team Number"
    }
)

fig.update_layout(
    xaxis_title="Lap Number",
    yaxis_title="Lap Time (s)",
    legend_title="Team Number"
)

fig.show()

In [ ]:
LeMans_filtered = LeMans[LeMans['pit_time'].isna() | (LeMans['pit_time'] == 0)]

fig = px.line(LeMans_filtered,
    x="lap_number",
    y="lap_time_s",
    title="Driver Stint Consistency by Team",
    color="driver_name",       
    line_group="driver_stint", 
    facet_col="team_no",          
    facet_col_wrap=2,          
    markers=True,
    labels={
        "lap_number": "Lap Number",
        "lap_time_s": "Lap Time (s)",
        "driver_name": "Driver",
        "team_no": "Team Number",
        "driver_stint": "Driver Stint",
    }
)

fig.update_layout(
    xaxis_title="Lap Number",
    yaxis_title="Lap Time (s)",
    legend_title="Driver",
    hovermode='closest',
    height=800  
)

fig.show()

## Pit Stop Strategy Effictiveness 

In [ ]:
LeMans["pit_time"] = pd.to_timedelta(LeMans["pit_time"], errors="coerce")
LeMans["pit_time_sec"] = LeMans["pit_time"].dt.total_seconds()

fig = go.Figure()

for team in LeMans["team_no"].unique():
    team_df = LeMans[LeMans["team_no"] == team]

    fig.add_trace(go.Scatter(
        x=team_df["lap_number"],
        y=team_df["lap_time_s"],
        mode="lines",
        line=dict(shape="hv"),
        name=f"Team {team}",
        legendgroup=str(team),  
        showlegend=True
    ))

    pit_laps = team_df[team_df["pit_time_sec"].notna() & (team_df["pit_time_sec"] > 0)]
    fig.add_trace(go.Scatter(
        x=pit_laps["lap_number"],
        y=pit_laps["lap_time_s"],
        mode="markers",
        marker=dict(size=10, symbol="x", color="red"),
        name="Pit Stop",
        legendgroup=str(team),  
        showlegend=False        
    ))

fig.update_layout(
    title="Pit Stop Strategy Effectiveness",
    xaxis_title="Lap Number",
    yaxis_title="Lap Time (s)",
    legend_title="Team Number"
)

fig.show()


## Stint Degradation

In [ ]:
LeMans_filtered = LeMans[LeMans['pit_time'].isna() | (LeMans['pit_time'] == 0)]

stint_summary = (
    LeMans_filtered.groupby(["driver_stint", "team_no", "driver_name"]).agg(
        start_lap=("lap_number", "min"),
        end_lap=("lap_number", "max"),
        stint_length=("lap_number", "count"),
        start_pace=("lap_time_s", lambda x: x.head(min(3, len(x))).mean()), 
        end_pace=("lap_time_s", lambda x: x.tail(min(3, len(x))).mean())
    )
    .reset_index()
)

stint_summary = stint_summary[stint_summary['stint_length'] >= 5]

stint_long = pd.concat([
    stint_summary[["driver_stint", "team_no", "start_lap", "start_pace"]].rename(columns={"start_lap": "lap", "start_pace": "lap_time"}),
    stint_summary[["driver_stint", "team_no", "end_lap", "end_pace"]].rename(columns={"end_lap": "lap", "end_pace": "lap_time"})], ignore_index=True)

fig = px.line(stint_long,
    x="lap",
    y="lap_time",
    color="team_no",
    line_group="driver_stint",
    markers=True,
    title="Stint Degradation: Start vs End Pace",
    labels={
        "lap": "Lap Number",
        "lap_time": "Lap Time (s)",
        "team_no": "Team",
        "driver_stint": "Driver Stint"
    }
)

fig.update_layout(
    xaxis_title="Lap Number",
    yaxis_title="Lap Time (s)",
    legend_title="Team"
)

fig.show()

In [ ]:
LeMans_filtered = LeMans[LeMans['pit_time'].isna() | (LeMans['pit_time'] == 0)]

LeMans_filtered["stint_lap_index"] = LeMans_filtered.groupby("driver_stint").cumcount()

def degradation_rate(stint_df):
    x = stint_df['stint_lap_index'].values
    y = stint_df['lap_time_s'].values
    
    if len(x) < 5:  
        return pd.Series({
            'degradation_rate': None,
            'team_no': stint_df['team_no'].iloc[0],
            'driver': stint_df['driver_name'].iloc[0],
            'stint_length': len(x)
        })
    
    slope = stats.linregress(x, y).slope
    
    return pd.Series({
        'degradation_rate': slope,  
        'team_no': stint_df['team_no'].iloc[0],
        'driver': stint_df['driver_name'].iloc[0],
        'stint_length': len(x)
    })

stint_summary = LeMans_filtered.groupby("driver_stint").apply(degradation_rate).reset_index()

stint_summary = stint_summary.dropna(subset=['degradation_rate'])

fig = px.scatter(
    stint_summary,
    x="stint_length",
    y="degradation_rate",
    color="team_no",
    hover_data={
        "driver": True,
        "driver_stint": True,
        "degradation_rate": ":.4f",
        "team_no": False
    },
    trendline="ols",
    title="Tire Degradation Rate vs Stint Length",
    labels={
        "stint_length": "Stint Length (laps)",
        "degradation_rate": "Degradation Rate (s/lap)",
        "team_no": "Team",
        "driver": "Driver",
        "driver_stint": "Driver Stint"
    }
)

fig.update_layout(
    xaxis_title="Stint Length (laps)",
    yaxis_title="Degradation Rate (seconds per lap)",
    legend_title="Team",
    hovermode='closest',
    height=600
)

fig.show()

## Gap Evolution to Leader

In [ ]:
fig = px.line(LeMans,
    x="lap_number",
    y="class_gap",
    color="team_no",
    title="Gap Evolution to Race Leader",
    labels={
        "lap_number": "Lap Number",
        "class_gap": "Gap to Leader (s)",
        "team_no": "Team Number"
    }
)

fig.show()

## Drive Pace Delta within Team and Team Pace Delta

In [ ]:
LeMans_filtered = LeMans[LeMans['pit_time'].isna() | (LeMans['pit_time'] == 0)]

LeMans_filtered["stint_lap_index"] = LeMans_filtered.groupby("driver_stint").cumcount()

baseline_pace = (
    LeMans_filtered[LeMans_filtered["stint_lap_index"] < 3]
    .groupby("driver_stint")["lap_time_s"]
    .mean()
)

LeMans_filtered["lap_time_delta"] = (
    LeMans_filtered["lap_time_s"] - LeMans_filtered["driver_stint"].map(baseline_pace)
)

fig = px.box(LeMans_filtered,
    x="team_no",  
    y="lap_time_delta",
    color="driver_name",  
    points="outliers",
    hover_data={
        'driver_name': False,  
        'lap_number': True,
        'stint_lap_index': True,
        'lap_time_s': ':.2f',
        'team': False
    },
    title="Driver Pace Delta Within Team",
    labels={
        "team_no": "Team Number",
        "lap_time_delta": "Lap Time Δ vs Stint First Lap (s)",
        "driver_name": "driver_name",  
        "lap_number": "Lap Number",
        "stint_lap_index": "Lap in Stint",
        "lap_time_s": "Actual Lap Time"
    }
)


fig.update_layout(
    xaxis_title="Team",
    yaxis_title="Lap Time Δ (s)",
    legend_title="driver_name", 
    height=600,
    hovermode='closest'
)

fig.show()

In [ ]:
LeMans_filtered = LeMans[LeMans['pit_time'].isna() | (LeMans['pit_time'] == 0)]

LeMans_filtered["stint_lap_index"] = LeMans_filtered.groupby("driver_stint").cumcount()

baseline_pace = (
    LeMans_filtered[LeMans_filtered["stint_lap_index"] < 3]
    .groupby("driver_stint")["lap_time_s"]
    .mean()
)

LeMans_filtered["lap_time_delta"] = (
    LeMans_filtered["lap_time_s"] - LeMans_filtered["driver_stint"].map(baseline_pace)
)

fig = px.violin(LeMans_filtered,
    x="team_no",
    y="lap_time_delta",
    box=True,
    points="outliers",  
    color="team",
    hover_data={
        'driver_name': True,
        'stint_lap_index': True,
        'lap_number': True,
        'lap_time_s': ':.2f',
        'team': False
    },
    title="Team Pace Comparison (Lap Time Δ vs Baseline)",
    labels={
        "team": "Team",
        "lap_time_delta": "Lap Time Δ (s)",
        "driver_name": "Driver",
        "stint_lap_index": "Lap in Stint",
        "lap_number": "Lap Number",
        "lap_time_s": "Actual Lap Time"
    }
)

fig.update_layout(
    xaxis_title="Team",
    yaxis_title="Lap Time Δ vs Baseline (s)",
    showlegend=False,
    height=600,
    hovermode='closest'
)

fig.show()